# QM9-wide atomic-environment novelty (6-Bohr classification)

Novelty of the QM9 remainder with the whole-QM9 6-Bohr classification of the environment-holdout study as the
reference. The distance of an atom to AIMEl is its distance to the 15th-nearest AIMEl atom of the same element in the
unscaled 6-Bohr PCA space; an atom is rare beyond the p99 of AIMEl's own atoms, each scored without its molecule.
Produces the dipole figure of the main text and the t-SNEs, fold maps and dipole-error molecules of the SI.

In [ ]:
import importlib
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = os.path.abspath(os.path.join('..', '..'))
for _p in (os.path.join(project_root, 'analysis', 'molecular'), os.path.join(project_root, 'analysis', 'inference')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import compare_atomic_envs_6b as cae6
importlib.reload(cae6)

%matplotlib inline
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)
FIGS = 'figs'
os.makedirs(FIGS, exist_ok=True)

## 1. Load the classification and check it

In [ ]:
labels = cae6.load_env_labels()
folds = cae6.load_env_folds()
atoms = cae6.atom_label_table(labels)
frame = cae6.load_frame()
n_checked = cae6.check_atom_order(frame, atoms)

print(f"molecules: {len(labels):,} ({int(labels['in_aimel'].sum()):,} AIMEl, {int((~labels['in_aimel']).sum()):,} remainder)")
print(f"atoms: {len(atoms):,}; label element == frame element for all {n_checked:,}")
summary = atoms.groupby('element').agg(atoms=('label', 'size'), labels=('label', lambda s: s[~s.str.endswith('_-1')].nunique()),
                                        noise_pct=('is_noise', lambda s: round(100 * s.mean(), 1)))
summary.loc[cae6.ELEMENTS]

## 2. Environments the remainder populates and AIMEl does not

In [ ]:
census = cae6.label_census(atoms, folds)
print('remainder-only labels:')
display(census[census['remainder_only']][['label', 'n_aimel', 'n_remainder', 'remainder_share', 'env_fold']])
print('most enriched in the remainder (labels with AIMEl atoms):')
display(census[~census['remainder_only']].nlargest(10, 'log2_enrichment')[
    ['label', 'n_aimel', 'n_remainder', 'aimel_share', 'remainder_share', 'log2_enrichment', 'env_fold']].round(4))
print('log2 enrichment per element:')
census.groupby('element')['log2_enrichment'].describe(percentiles=[0.05, 0.5, 0.95]).loc[cae6.ELEMENTS].round(2)

## 3. Per-atom label support (descriptive)

In [ ]:
atom_nov, calib = cae6.support_novelty(atoms)
calib

## 4. Distance to AIMEl (the novelty score)

In [ ]:
pca = cae6.soap_pca_6b(frame)
display(cae6.pca_summary(pca))

_p = os.path.join(cae6.CACHE_DIR, 'knn_pca_6b.pkl')
if os.path.exists(_p):
    atom_dist, dist_calib = pd.read_pickle(_p)
else:
    atom_dist, dist_calib = cae6.knn_novelty_pca(pca, atoms, knn=15)    # every QM9 atom; AIMEl leave-one-molecule-out
    pd.to_pickle((atom_dist, dist_calib), _p)
display(dist_calib)

rem_nov = atom_nov[~atom_nov['in_aimel']]
rem_dist = atom_dist[~atom_dist['in_aimel']]
cae6.novelty_agreement(rem_nov, rem_dist, names=('label', 'dist6'))

## 5. Per-molecule novelty

The AIMEl row is the chance level: 1 % of each element's atoms are rare by construction.

In [ ]:
mol_nov = cae6.molecule_novelty_6b(atom_nov, frame)                  # label facts: noise, remainder-only labels, support
mol_dist = cae6.molecule_distance_novelty(atom_dist)                 # the novelty score: 6-Bohr distance to AIMEl (§4)
mol = mol_nov.merge(mol_dist.drop(columns='in_aimel'), on='index', validate='one_to_one')
rem_mol = mol_nov[~mol_nov['in_aimel']]
display(cae6.molecule_baseline(mol, count_col='n_rare_dist', pct_col='max_dist_pct'))
(mol[~mol['in_aimel']].sort_values(['n_rare_dist', 'max_ratio'], ascending=False)
 .head(10)[['index', 'smiles', 'n_atoms', 'n_rare_dist', 'frac_rare_dist', 'max_ratio', 'n_novel', 'n_noise']])

## 6. The environment folds of the holdout study

In [ ]:
cae6.env_fold_novelty(mol, labels, folds, count_col='n_rare_dist')

## 7. t-SNE views

20,000 AIMEl and 20,000 remainder items (perplexity 50, PCA initialisation, seed 42); difference panels: log2
enrichment of the remainder over AIMEl per hexbin. Text and font sizes are arguments of each call.

### 7.1 Chemotypes

In [ ]:
chemotypes = pd.read_pickle(cae6.CHEMOTYPES_PKL)
display(cae6.chemotype_table(mol, chemotypes, count_col='n_rare_dist').sort_values('remainder: >=1 rare %', ascending=False).head(8).round(4))
emb_chem = cae6.chemotype_embedding(frame, chemotypes)

# ── style and text of the figures of this notebook (edit for the paper) ──
FONTS = {'title': 18, 'label': 16, 'tick': 12, 'legend': 12, 'annot': 12}   # QT-Net's published sizes
TITLE_WEIGHT, LABEL_WEIGHT = 'normal', 'normal'

# printed ~5.2 in wide in the paper: ~9 / 8 / 7 pt; the chemotype-coloured panel is left out of the paper figure
FONTS_CHEM = {'title': 19, 'label': 17, 'tick': 15, 'legend': 15, 'annot': 15}
cae6.tsne_chemotype_view(emb_chem, titles=('Chemotypes', 'AIMEl', 'QM9 remainder', 'Difference'),
                         cbar_labels=('AIMEl molecules', 'QM9 remainder molecules',
                                      'log$_2$ enrichment'),          # of the remainder over AIMEl (caption)
                         xlabel='t-SNE 1', ylabel='t-SNE 2', fonts=FONTS_CHEM, title_weight=TITLE_WEIGHT,
                         label_weight=LABEL_WEIGHT, chemotype_panel=False, figsize=(11.0, 4.9),
                         save_path=f'{FIGS}/tsne_chemotypes_6b.png')
plt.show()

### 7.2 Atomic environments, per element

In [ ]:
emb_env = {}
FONTS_ENV = {'title': 19, 'label': 17, 'tick': 15, 'legend': 15, 'annot': 15}   # ~9 / 8 / 7 pt printed 5.2 in wide
for el in cae6.ELEMENTS:
    emb_env[el] = cae6.env_embedding(pca, atom_nov, el)
    # the element is named in the caption, not in the panel titles
    cae6.tsne_env_view(emb_env[el], el, title_fmt='{panel}', panels=('AIMEl', 'QM9 remainder', 'Difference'),
                       cbar_labels=('AIMEl atoms', 'QM9 remainder atoms', 'log$_2$ enrichment'),
                       xlabel='t-SNE 1', ylabel='t-SNE 2', fonts=FONTS_ENV, title_weight=TITLE_WEIGHT,
                       label_weight=LABEL_WEIGHT, figsize=(11.0, 4.9), save_path=f'{FIGS}/tsne_env_6b_{el}.png')
    plt.show()

### 7.3 Test panels of the first repeat

In [ ]:
# ── style and text of the panel maps (edit for the paper) ──
FOLD_COLORS = ['#9900cc', '#0000ff', '#c9a400', '#008066', '#b40426']   # purple, winter, mustard, summer, coolwarm
BG_LABELS = ('AIMEl denser', 'QM9 remainder denser')                    # background entries of the legends
BG_LABELS_CHEM = ('', 'QM9 remainder denser')    # chemotype map: the fold hexbins cover every AIMEl region ('' = no entry)

if os.path.isdir(cae6.RUNS_DIR):
    panels = cae6.load_fold_panels(range(5))      # folds 0-4 = repeat 0
    display(panels.groupby(['fold', 'cv_fold', 'env_fold', 'panel']).size().unstack('panel'))
    cae6.tsne_panel_map(emb_chem, panels, colors=FOLD_COLORS, gridsize=45,
                        title=None,                          # the caption says what the map shows
                        fold_fmt='Fold {fold}', dark_label='Dark: mostly panel B', light_label='Light: mostly panel A',
                        bg_labels=BG_LABELS_CHEM, legend_ncol=3, xlabel='t-SNE 1', ylabel='t-SNE 2',
                        fonts={'title': 20, 'label': 18, 'tick': 16, 'legend': 16, 'annot': 16},   # ~7-8 pt printed 4.1 in wide
                        title_weight=TITLE_WEIGHT, label_weight=LABEL_WEIGHT, figsize=(9.0, 10.0),
                        save_path=f'{FIGS}/tsne_panels_chemotypes.png')
    plt.show()
    cae6.tsne_env_panel_map(emb_env, panels, colors=FOLD_COLORS, gridsize=45, title_fmt='{element}',
                            suptitle=None,                   # the caption says what the maps show
                            fold_fmt='Fold {fold}: {env_labels}', dark_label='Dark: mostly in panel-B molecules',
                            light_label='Light: mostly in panel-C molecules', bg_labels=BG_LABELS, legend_ncol=2,
                            legend_gap=14.0,                     # blank space (pt) between the x labels and the legend
                            xlabel='t-SNE 1', ylabel='t-SNE 2',
                            fonts={'title': 23, 'label': 20, 'tick': 18, 'legend': 18, 'annot': 18},   # ~8-10 pt printed 5.2 in wide
                            title_weight='bold',             # the element names
                            label_weight=LABEL_WEIGHT,
                            w_pad=0.15, h_pad=0.15, wspace=0.06, hspace=0.06,   # space between the four maps
                            figsize=(13.0, 15.0),
                            save_path=f'{FIGS}/tsne_panels_environments.png')
    plt.show()
else:
    print(f'[skip] {cae6.RUNS_DIR} not found')

## 8. QT-Net dipole error against the distance to AIMEl

In [ ]:
import analyse_inference as ai
importlib.reload(ai)

inferred = pd.read_pickle(cae6.INFERRED_PKL)
df_mu = ai.load_dipole_preds(cae6.INFERRED_PKL)                      # 'mu' (QM9 reference), 'mu_inferred' (QT-Net)
dipole_metrics = ai.dipole_metric_table(df_mu, n_boot=1000, seed=0)
err = cae6.dipole_errors(df_mu, rem_mol, mol_dist)                  # novelty: 6-Bohr distance to AIMEl (§4)
err = pd.concat([err, cae6.outlier_groups(err['smiles'])], axis=1)   # zwitterion types, oxabicyclopentanols (§8.1)
n_aimel = cae6.outlier_groups(frame.loc[frame['in_aimel'], 'smiles']).sum()   # the same groups in AIMEl

# ── style and text (edit for the paper) ──
NOVELTY_COLORS = ('#3b4cc0', '#f39c7d', '#b40426')                 # same blue → red as the 'error relative' plots
HIGHLIGHTS = [dict(col='protonated_zwitterion', label='Protonated zwitterions', color='#000000'),
              dict(col='other_zwitterion', label='Other zwitterions', color='#8c8c8c'),
              dict(col='oxabicyclopentanol', label='Alkyl-oxabicyclopentanols', color='#1a9850')]
err['pct_rare_dist'] = 100 * err['frac_rare_dist']
fig = cae6.dipole_figure(err, dipole_metrics, right='parity',        # or right='error_vs_distance'
                         color_col='pct_rare_dist', count_col='n_rare_dist', rare_hi=3,   # 6-Bohr distance to AIMEl (§4)
                         group_labels=('No rare', '1–2 rare', '≥3 rare'),
                         group_fmt='{label}: MAE {mae:.3f} D',          # the group sizes go to the caption
                         highlights=HIGHLIGHTS, highlight_min_err=0.0,   # 0: outline all members
                         highlight_lw=0.5, highlight_alpha=1.0,          # outline width and opacity (per group: 'lw', 'alpha')
                         highlight_fmt='{label}', legend_fontsize=11,   # 11: the legend stays above the data (< 17 D)
                         bins=80, cmap_count='winter', novelty_colors=NOVELTY_COLORS, novelty_range=None,
                         xlabel=r'Target molecular $|\mathbf{\mu}|$ [D]',
                         ylabel=r'Molecular $|\mathbf{\mu}|$ from QT-Net [D]',
                         count_label='Count', novelty_label='Rare atoms (%)',
                         cbar_box=(0.50, 0.09, 0.46, 0.035), cbar_label_size=11,   # bottom-right corner, clear of the data
                         legend_kw=dict(labelspacing=0.15, borderpad=0.3, borderaxespad=0.25),   # legend clear of the data (< 17 D)
                         titles=('', ''), title_weight=TITLE_WEIGHT, label_weight=LABEL_WEIGHT,
                         fonts={'title': 18, 'label': 15.5, 'tick': 13, 'legend': 12, 'annot': 13},   # printed 0.85 \linewidth:
                         figsize=(9.6, 4.8),                                                          # ~8.5 / 7 / 6.5 pt
                         save_path=f'{FIGS}/dipole_parity_novelty_6b.pdf')
plt.show()
# caption numbers for the outlines: group size, members off by > 1 D, and the same group in AIMEl
display(pd.DataFrame([{'group': h['label'], 'outline': h['color'], 'n_remainder': int(err[h['col']].sum()),
                       'n_off_by_>1_D': int((err[h['col']] & (err['abs_err'] > 1.0)).sum()),
                       'n_AIMEl': int(n_aimel[h['col']])} for h in HIGHLIGHTS]))
print('MAE by number of atoms beyond AIMEl\'s p99 6-Bohr distance (the marginals of the right panel):')
display(cae6.dipole_group_table(err, 'n_rare_dist', exclude_col='zwitterion').round(3))
_sq = (err['mu_inferred'] - err['mu']) ** 2
_r2 = lambda m: 1 - _sq[m].sum() / ((err.loc[m, 'mu'] - err.loc[m, 'mu'].mean()) ** 2).sum()
_all, _zw = np.ones(len(err), bool), err['zwitterion'].to_numpy()
for _thr in (1, 3, 5):
    _m = err['abs_err'] > _thr
    print(f"off by > {_thr} D: {int(_m.sum()):,} molecules, {100 * (err.loc[_m, 'n_rare_dist'] > 0).mean():.1f} % with ≥ 1 atom far "
          f"from AIMEl, {100 * (err.loc[_m, 'n_rare_dist'] >= 3).mean():.1f} % with ≥ 3")
print(f"R² all {_r2(_all):.4f} | without zwitterions {_r2(~_zw):.4f} | zwitterions only {_r2(_zw):.4f} | "
      f"zwitterions' share of the squared error {100 * _sq[_zw].sum() / _sq.sum():.1f} %")

### 8.1 The largest errors

In [ ]:
outliers = cae6.dipole_outliers(err, n=20)
display(outliers.round(3))
display(cae6.charge_separation_table(err, frame, top=100).round(3))
display(cae6.zwitterion_table(err, frame, flag_col='zwitterion', min_err=1.0).round(2))
display(cae6.outlier_group_table(err, frame, min_err=1.0).round(3))
display(cae6.error_threshold_table(err, thresholds=(0.5, 1, 2, 3, 5)).round(1))
_r3 = err['n_rare_dist'] >= 3
print(f"≥ 3 rare atoms: {int(_r3.sum()):,} molecules, {int((_r3 & ~err['alkyne'] & ~err['zwitterion']).sum()):,} "
      'of them neither alkynes nor zwitterions')
_big = err[err['abs_err'] > 1]
for _g in ('alkyne', 'oxabicyclopentanol'):
    _over = _big.loc[_big[_g], 'mu_inferred'] > _big.loc[_big[_g], 'mu']
    print(f"{_g}: |Δμ| > 1 D in {len(_over)} molecules, overestimated in {int(_over.sum())} ({_over.mean():.3f})")
# n_rare is the label-support count, n_rare_dist the distance one; the appendix's 'without rare atoms' refers to n_rare
_far = err[err['abs_err'] > 5]
print(f"{len(_far)} molecules with |Δμ| > 5 D, {int(_far['zwitterion'].sum())} of them zwitterions; the others:")
display(_far[~_far['zwitterion']][['index', 'smiles', 'mu', 'mu_inferred', 'abs_err', 'n_rare', 'n_rare_dist',
                                   'max_ratio', 'n_noise']].round(3))
from rdkit import Chem
_cage = Chem.MolFromSmarts('C12CC(O1)C2')                           # 2-oxabicyclo[1.1.1]pentane
_has = frame['smiles'].map(lambda smi: Chem.MolFromSmiles(smi).HasSubstructMatch(_cage)).to_numpy()
print(f"2-oxabicyclo[1.1.1]pentanes: {int((_has & frame['in_aimel']).sum())} in AIMEl, "
      f"{int((_has & ~frame['in_aimel']).sum())} in the remainder")
_motif = Chem.MolFromSmarts('[CX4;R]([OX2H1])([CX4;!R])1[CX4H1;R]2[CX4H2;R][CX4H1;R]1O2')   # alkyl + OH on the bridge C
_hm = frame['smiles'].map(lambda smi: Chem.MolFromSmiles(smi).HasSubstructMatch(_motif)).to_numpy()
print(f"... with an alkyl chain and an OH on the bridge carbon: {int((_hm & frame['in_aimel']).sum())} in AIMEl "
      f"({', '.join(frame.loc[_hm & frame['in_aimel'], 'smiles'])}), {int((_hm & ~frame['in_aimel']).sum())} in the remainder")

spread = cae6.ensemble_spread(inferred)                               # QT-Net ensemble disagreement per molecule
_e = err.merge(spread, on='index', validate='one_to_one')
display(_e[_e['index'].isin(_far.loc[~_far['zwitterion'], 'index'])][['index', 'smiles', 'abs_err', 'max_atom_mu_std',
                                                                       'max_atom_mu_std_pct']].round(2))
for _name, _m in (('all molecules', np.ones(len(_e), bool)), ('without zwitterions', ~_e['zwitterion'].to_numpy())):
    _c = _e.loc[_m, ['abs_err', 'max_atom_mu_std', 'frac_rare_dist', 'max_ratio', 'frac_rare']].corr('spearman')['abs_err']
    print(f"Spearman ρ of |Δμ| ({_name}): ensemble spread {_c['max_atom_mu_std']:.2f} | fraction of atoms far from "
          f"AIMEl {_c['frac_rare_dist']:.2f} | largest atom distance {_c['max_ratio']:.2f} | fraction of label-rare "
          f"atoms {_c['frac_rare']:.2f}")


### 8.2 One molecule in 3D

Set `GROUP` and `RANK`, or `MOLECULE` to any QM9 id.

In [ ]:
GROUP = 'protonated_zwitterion'   # 'protonated_zwitterion', 'other_zwitterion', 'oxabicyclopentanol', 'alkyne' or None
RANK = 0                           # 0 = the group's largest error, 1 = the second largest, ...
MOLECULE = None                    # or any QM9 id of the remainder (overrides GROUP / RANK)
THRESHOLD_PCT = 99.0               # distance percentile: CPK at or below, held-out colour above
LABELS = None                      # or 'label', 'distance' (percentile), 'index'

mol = cae6.render_molecule(err, atom_nov, inferred, atom_dist, group=GROUP, rank=RANK, index=MOLECULE,
                           color_by='distance', near_colors=cae6.CPK, far_colors=cae6.FAR_COLORS,
                           threshold_pct=THRESHOLD_PCT, labels=LABELS, label_hydrogens=False, width=560, height=440,
                           title_fmt='QM9 {index}', fonts=FONTS, title_weight=TITLE_WEIGHT,
                           save_path=f'{FIGS}/molecule_{GROUP}_{RANK}.png')
display(mol['env'].round(3))
mol['view'].show()
plt.show()

**Figure: the molecules behind the largest errors.** `screenshot_html` needs a Chromium-based browser; without one,
save the screenshots by hand under the same names and run the last line.

In [ ]:
PICKS = [dict(group='alkyne', rank=0), dict(group='protonated_zwitterion', rank=0, rotate=[(90, 'y')]),
         dict(group='other_zwitterion', rank=0),
         dict(group='oxabicyclopentanol', rank=0, rotate=[(-40, 'x'), (30, 'y')])]   # views chosen by eye
NEAR = {'H': '#FFFFFF', 'C': '#B4B4B4', 'N': '#778EFE', 'O': '#FF0D0D'}   # the paper's Fig. 1 'other atoms'
pngs = []
for k, pk in enumerate(PICKS):
    idx = cae6.pick_group_molecule(err, pk['group'], pk['rank'])
    env = cae6.molecule_environment_table(idx, atom_nov, inferred, atom_dist)
    html = cae6.write_molecule_html(env, idx, inferred, f'{FIGS}/molecules/{k}_{idx}.html', near_colors=NEAR,
                                    rotate=pk.get('rotate', ()))
    pngs.append(f'{FIGS}/molecules/{k}_{idx}.png')
    try:
        cae6.screenshot_html(html, pngs[-1])
    except (FileNotFoundError, OSError, subprocess.CalledProcessError, subprocess.TimeoutExpired) as e:
        print(f'[screenshot by hand] {html} -> {pngs[-1]} ({e})')
if all(os.path.exists(p) for p in pngs):
    cae6.tile_images(pngs, f'{FIGS}/dipole_error_molecules.png')